# EuroCropML x OLMoEarth Benchmark - Colab Runner

Runs the full benchmark pipeline on Google Colab with GPU.

Weights auto-download from HuggingFace. No file upload needed.

In [ ]:
# Cell 1: Setup
!git clone https://github.com/YOUR_USERNAME/eurocrop-olmoearth-benchmark.git
%cd eurocrop-olmoearth-benchmark
!pip install -r requirements.txt huggingface_hub -q

In [ ]:
# Cell 2: Write cloud config
import yaml

config = {
    'data': {
        'mode': 'cloud',
        'cloud_data_dir': './data',
        'cloud_country': 'Estonia',
        'top_n_classes': 15,
        'test_split': 0.2,
        'random_seed': 42
    },
    'model': {
        'mode': 'cloud',
        'cloud_model_id': 'allenai/OlmoEarth-v1_1-Nano',
        'batch_size': 32,
        'device': 'cuda'
    },
    'fewshot': {
        'shots': [10, 25, 50, 100],
        'n_repeats': 5
    },
    'output': {
        'metrics_dir': './results/metrics',
        'figures_dir': './results/figures'
    }
}

with open('config.yaml', 'w') as f:
    yaml.dump(config, f)
print('Cloud config written')

In [ ]:
# Cell 3: Download EuroCropML data (Estonia)
!eurocropsml download --country Estonia --output-dir ./data

In [ ]:
# Cell 4: Phase 1 - Classical baselines
!python experiments/phase1_baselines.py

In [ ]:
# Cell 5: Phase 2 - OLMoEarth embeddings (weights download from HF automatically)
!python experiments/phase2_embeddings.py

In [ ]:
# Cell 6: Phase 3 - Few-shot comparison
!python experiments/phase3_fewshot.py

In [ ]:
# Cell 7: Visualizations
import json
import matplotlib.pyplot as plt
import os

if os.path.exists('results/metrics/phase3_fewshot.json'):
    from src.viz.fewshot_curve import plot_fewshot_curve
    with open('results/metrics/phase3_fewshot.json') as f:
        results = json.load(f)
    plot_fewshot_curve(results, 'results/figures/fewshot_curve.png')
    plt.show()

if os.path.exists('results/metrics/emb_train.npy'):
    import numpy as np
    from src.viz.umap_viz import plot_umap
    from src.data.loader import load_dataset
    emb = np.load('results/metrics/emb_train.npy')
    X, y, _ = load_dataset('./data', 'Estonia', top_n_classes=15)
    plot_umap(emb[:len(y)], y, save_path='results/figures/umap_embeddings.png')
    plt.show()

In [ ]:
# Cell 8: Download results
from google.colab import files
import glob

for f in glob.glob('results/metrics/*.json'):
    files.download(f)
for f in glob.glob('results/figures/*.png'):
    files.download(f)
print('All results downloaded')